Code for machine learning and deep learning papers

In [1]:
import requests
import pandas as pd
import time

URL = "https://api.semanticscholar.org/graph/v1/paper/search"

QUERY = "machine learning deep learning"
TOTAL = 200
LIMIT = 20       # SAFE limit
OFFSET = 0
DELAY = 3        # seconds

rows = []
serial = 1

while len(rows) < TOTAL:
    params = {
        "query": QUERY,
        "limit": LIMIT,
        "offset": OFFSET,
        "fields": "title,abstract"
    }

    r = requests.get(URL, params=params)

    if r.status_code == 429:
        print("⏳ Rate limit hit. Waiting 10 seconds...")
        time.sleep(10)
        continue

    if r.status_code != 200:
        print("HTTP Error:", r.status_code)
        break

    data = r.json()
    papers = data.get("data", [])

    if not papers:
        break

    for paper in papers:
        title = paper.get("title")
        abstract = paper.get("abstract")

        if title and abstract:
            rows.append({
                "serial": serial,
                "title": title.strip(),
                "abstract": abstract.strip()
            })
            serial += 1

        if len(rows) >= TOTAL:
            break

    OFFSET += LIMIT
    time.sleep(DELAY)

df = pd.DataFrame(rows)
df.to_csv("ml_dl_papers.csv", index=False, encoding="utf-8")

print(f"✅ Successfully saved {len(rows)} papers")

⏳ Rate limit hit. Waiting 10 seconds...
⏳ Rate limit hit. Waiting 10 seconds...
HTTP Error: 500
✅ Successfully saved 0 papers


Code for NON-AI / NON-ML papers

In [ ]:
import requests
import pandas as pd
import time

URL = "https://api.semanticscholar.org/graph/v1/paper/search"

QUERY = "study OR analysis OR experiment OR investigation"

FIELDS_OF_STUDY = (
    "Biology,Medicine,Chemistry,Physics,Economics,"
    "Environmental Science,Materials Science,"
    "Sociology,Political Science"
)

TOTAL = 200
LIMIT = 20
OFFSET = 0
DELAY = 3

rows = []
serial = 1

while len(rows) < TOTAL:
    params = {
        "query": QUERY,
        "fieldsOfStudy": FIELDS_OF_STUDY,
        "limit": LIMIT,
        "offset": OFFSET,
        "fields": "title,abstract,year,venue,fieldsOfStudy"
    }

    r = requests.get(URL, params=params)

    if r.status_code == 429:
        print("⏳ Rate limit hit. Waiting 10 seconds...")
        time.sleep(10)
        continue

    if r.status_code != 200:
        print("HTTP Error:", r.status_code)
        break

    data = r.json()
    papers = data.get("data", [])

    if not papers:
        break

    for paper in papers:
        fos = paper.get("fieldsOfStudy") or []  # <-- FIX: If None, use empty list

        # Extra safety check (DOUBLE FILTER)
        if "Computer Science" in fos:
            continue

        title = paper.get("title")
        abstract = paper.get("abstract")

        if title and abstract:
            rows.append({
                "serial": serial,
                "title": title.strip(),
                "abstract": abstract.strip()
            })
            serial += 1

        if len(rows) >= TOTAL:
            break

    OFFSET += LIMIT
    time.sleep(DELAY)

df = pd.DataFrame(rows)
df.to_csv("non_ai_non_ml_papers.csv", index=False, encoding="utf-8")

print(f"✅ Saved {len(rows)} NON-AI / NON-ML papers")